### Make a temporary space in /flash

In [1]:
import os, random, string

In [2]:
cwd = os.path.basename(os.getcwd())
tmpdir = os.path.join("/flash/ReiterU/tmp/", 'tomo_' + ''.join(random.choices(string.ascii_uppercase, k=12)))
print(tmpdir)
if not os.path.exists(tmpdir):
    os.makedirs(tmpdir)

/flash/ReiterU/tmp/tomo_KODVMRYMCZNN


### PARAMETERS

In [3]:
#### path to relevant files
rootdir = "/bucket/ReiterU/OLdev/experiments/20250520_squid_cal520"
recname = "rec9"
savedir = "/bucket/ReiterU/OLdev/analysis/20250528_IGL_monitor_flip_analysis/20250520_rec9"

templatefile = os.path.join(rootdir, "2P", recname, "run2/mcorr_nonrigid/QC/QC_nonrigid/template.tif")
roifile = os.path.join(rootdir, "2P", recname, "ROI_yutaka_renamed.zip")
dfffile = os.path.join(rootdir, "2P", recname, "dff_ROI_yutaka/DFF.hdf5")
syncfile = os.path.join(rootdir, "sync", f"{recname}.tdms")

QI_THRESH = 0.25

### Import stuff

In [4]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from scipy import signal
import h5py
import subprocess

In [5]:
import sys
sys.path.append("../../../")
from calfuncs.io import read_sync_ttl
from calfuncs.full_field_step import get_ett2, plot_chirp_eta, compute_qi

In [6]:
from matplotlib import font_manager
import matplotlib as mpl

font_manager.fontManager.addfont('/bucket/ReiterU/common/fonts/arial.ttf')
mpl.rcParams.update({
    'font.size': 7,
    'font.family': 'sans-serif',
    'font.sans-serif': "Arial",
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'axes.linewidth': 0.5,
    'xtick.major.width': 0.5,
    'xtick.major.size': 2,
    'xtick.minor.size': 2,
    'xtick.minor.width': 0.5,
    'ytick.major.width': 0.5,
    'ytick.minor.width': 0.5,
    'ytick.major.size': 2,
    'ytick.minor.size': 2,
})

### Load TDMS file

In [7]:
sync_ttl = read_sync_ttl(syncfile)
twop_ttl = sync_ttl["twop_ttl"]
twop_re = sync_ttl["twop_re"]
vis_re = sync_ttl["vis_re"]
print(twop_ttl.shape, twop_re.shape, vis_re.shape)

(719500,) (10949,) (60,)


### Load DF/F

In [8]:
with h5py.File(dfffile, "r") as hf:
    F = hf["F"][:]
    dff = hf["dff"][:]
print(F.shape) # F is sampled at 1 kH

(166, 719500)


### Chirp

In [9]:
E = []
for nid in np.arange(0, F.shape[0]):
    ett1 = get_ett2(F, nid, vis_re[0::6], window=[-2000, 19000], baseline_window=[-4000, 70000], subsampling=50)
    ett1["session"] = 0
    ett2 = get_ett2(F, nid, vis_re[2::6], window=[-2000, 19000], baseline_window=[-27000, 43000], subsampling=50)
    ett2["session"] = 1
    ett3 = get_ett2(F, nid, vis_re[4::6], window=[-2000, 19000], baseline_window=[-51000, 19000], subsampling=50)
    ett3["session"] = 2
    ett = pd.concat([ett1, ett2, ett3])
    ett["neuron"] = nid
    ett = ett.set_index(["neuron", "session", "trial", "time"])
    E.append(ett)
E = pd.concat(E)

In [ ]:
dt = 1e-2
stimulus = np.zeros(int(21.0/dt), dtype=np.float32)
stimulus[0:int(2.0/dt)] = 0
stimulus[int(2.0/dt):int(6.0/dt)] = 1
stimulus[int(6.0/dt):int(10.0/dt)] = 0
stimulus[int(10.0/dt):int(12.0/dt)] = 0.5
stimulus[int(12.0/dt):int(20.0/dt)] = 0.5 + 0.5 * signal.chirp(np.linspace(0, 8, int(20.0/dt)-int(12.0/dt)), f0=0.5, f1=10, t1=8, method='logarithmic', phi=90)
stimulus[int(20.0/dt):int(21.0/dt)] = 0

for nid in np.arange(0, dff.shape[0]):
    qi = compute_qi(E, nid)
    print("Neuron:", nid, "QI:", qi)
    savename = os.path.join(tmpdir, f"ETA_{nid:03d}.png")
    plot_chirp_eta(E, nid, stimulus, vmin=-2, vmax=3,  cmap_name="bone_r", figsize=(5,2.0), show_plot=True, figsavename=savename)

### Compute trial-average

In [11]:
R = []
neurons = E.index.get_level_values("neuron").unique()

for n in neurons:
    M = []
    df2 = E.xs(n, level="neuron")
    for i in range(3):
        df3 = df2.xs(i, level="session")
        mat = df3.reset_index().pivot(index="trial", columns="time", values="y").to_numpy()
        M.append(mat)
    M = np.hstack(M)
    M = M[~np.all(np.isnan(M), axis=1)] # remove trials with NaN values
    qi = compute_qi(E, n)
    y_mean = np.nanmean(M, axis=0)
    row = {
        "recording": 0,
        "neuron": n,
        "qi": qi,
    }
    for j, value in enumerate(y_mean):
        row[f"time_{j}"] = value

    R.append(row)

R = pd.DataFrame(R)

In [12]:
display(R)

,recording,neuron,qi,time_0,time_1,time_2,time_3,time_4,time_5,time_6,...,time_1250,time_1251,time_1252,time_1253,time_1254,time_1255,time_1256,time_1257,time_1258,time_1259
0,0,0,0.184535,-0.496723,-0.064244,0.100119,-0.513608,-0.651167,-0.374586,-0.245436,...,-1.105477,-0.671642,-0.287771,-0.083864,-0.562125,-0.669846,-0.888059,-0.769872,-0.560761,-0.349786
1,0,1,0.176723,-0.338148,-0.002645,0.062156,-0.278338,-0.244179,-0.052461,-0.317979,...,-0.232773,-0.186190,-0.353516,-0.178961,-0.510898,-0.365672,-0.156577,0.128295,-0.013497,-0.121863
2,0,2,0.161234,-0.277970,-0.133686,-0.281144,-0.474864,-0.286390,-0.618958,-0.276428,...,-0.267799,-0.346501,-0.439632,-0.535179,-0.671075,-0.417483,-0.221191,-0.315232,-0.533158,-0.329472
3,0,3,0.245547,0.479401,0.360607,0.423554,0.511326,0.767199,0.708574,0.283274,...,-0.865355,-1.006896,-0.542468,-0.889487,-0.765179,-0.604457,-0.489900,-0.665676,-0.760878,-0.787853
4,0,4,0.243079,-0.017643,-0.302466,0.142755,-0.003256,0.024804,-0.359306,-0.067421,...,-0.029812,0.309738,0.650658,0.818703,0.607710,0.684958,0.096348,-0.365864,0.266310,0.326363
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
161,0,161,0.463819,0.944814,0.813181,1.034440,1.170720,1.010375,1.179510,1.068853,...,-0.953082,-0.676003,-0.924428,-1.183434,-1.352273,-1.207489,-1.311421,-1.161894,-0.946541,-0.880675
162,0,162,0.292457,-0.033433,0.017442,-0.360674,-0.382254,-0.500314,-0.364157,-0.236925,...,-0.509305,-0.496653,-0.547676,-0.309114,-0.401800,-0.422528,-0.238449,-0.371827,-0.280560,-0.352066
163,0,163,0.396620,-0.435439,-0.635920,-0.919587,-0.743097,-0.464459,-0.490793,-0.795459,...,-0.747660,-0.867171,-0.871526,-0.439979,-0.555211,-0.484942,-0.812623,-0.836721,-0.684676,-0.818907
164,0,164,0.239983,0.504380,0.645896,0.857357,0.683809,0.795009,1.181959,1.028210,...,-0.844483,-0.472968,-0.248824,-0.334899,-0.460923,-0.692259,-0.339115,0.047618,-0.523326,-0.515242


In [13]:
with pd.HDFStore(os.path.join(tmpdir, "response_table.hdf5"), mode='w') as store:
    store.put('E', E)
    store.put('responses', R)

### Copy data to bucket

In [14]:
cmd1 = f'ssh deigo "mkdir -p {savedir}"'
cmd2 = f'ssh deigo "cp -r {tmpdir}/* {savedir}"'
print(cmd1)
print(cmd2)

subprocess.run(cmd1, shell=True, check=True)
subprocess.run(cmd2, shell=True, check=True)

ssh deigo "mkdir -p /bucket/ReiterU/OLdev/analysis/20250528_IGL_monitor_flip_analysis/20250520_rec9"
ssh deigo "cp -r /flash/ReiterU/tmp/tomo_KODVMRYMCZNN/* /bucket/ReiterU/OLdev/analysis/20250528_IGL_monitor_flip_analysis/20250520_rec9"


Offending key for IP in /home/t/tomoyuki-mano1/.ssh/known_hosts:2
Matching host key in /home/t/tomoyuki-mano1/.ssh/known_hosts:6
Offending key for IP in /home/t/tomoyuki-mano1/.ssh/known_hosts:2
Matching host key in /home/t/tomoyuki-mano1/.ssh/known_hosts:6


CompletedProcess(args='ssh deigo "cp -r /flash/ReiterU/tmp/tomo_KODVMRYMCZNN/* /bucket/ReiterU/OLdev/analysis/20250528_IGL_monitor_flip_analysis/20250520_rec9"', returncode=0)